# Aula Semana 03: EDA Avançada (Fase 2) & Preparação dos Dados (Fase 3)

## 1. Visão Geral
Nesta aula, concluiremos a **Fase 2 (Data Understanding)** com a Análise Exploratória de Dados (EDA) avançada utilizando medidas de tendência central, dispersão e gráficos Seaborn, e executaremos a **Fase 3 (Data Preparation)** no mesmo dataset de risco de crédito (`CRISP_DM_Random_Forest_Credito.ipynb`).

---

## 2. Parte I: Análise Exploratória de Dados (EDA)

### Medidas de Tendência Central vs. Dispersão
- **Média vs. Mediana**: Em variáveis assimétricas como `renda_mensal`, a média é distorcida por altos salários. A mediana fornece o centro real.
- **Intervalo Interquartil (IQR)**: Diferença entre o percentil 75% ($Q_3$) e o percentil 25% ($Q_1$). Define a caixa do Boxplot.

---

## 3. Parte II: CRISP-DM Fase 3 — Pipeline de Data Preparation

O pipeline de tratamento transforma dados brutos e ruidosos em um conjunto de matrizes limpas ($X, y$):

1. **Padronização de Strings**: `.str.strip().str.upper()`
2. **Sanitização de Outliers Espúrios**: Substituição de idades inválidas (`< 18` ou `> 100`) e rendas absurdamente altas por `np.nan`.
3. **Engenharia de Atributos (Feature Engineering)**:
   - `valor_parcela = valor_solicitado / numero_parcelas`
   - `comprometimento_renda = valor_parcela / (renda_mensal + 1e-5)`
4. **Imputação de Valores Nulos**: Preenchimento pela Mediana em colunas numéricas (`.fillna()`).
5. **Descarte de Ruídos e Identificadores**: `.drop(columns=['proponente_id', 'ip_origem_hash', ...])`.
6. **Codificação One-Hot Encoding**: `pd.get_dummies(..., drop_first=True)`.
7. **Divisão Treino/Teste**: `train_test_split(..., test_size=0.25, stratify=y)`.


In [ ]:
# Execução completa do Pipeline EDA e Data Preparation em Python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

# Reutilizando df_raw gerado na Semana 02
df_clean = df_raw.copy()

# Step 1: Padronização de strings
df_clean['tipo_vinculo'] = df_clean['tipo_vinculo'].astype(str).str.strip().str.upper()

# Step 2: Sanitização de outliers espúrios
df_clean.loc[(df_clean['idade'] < 18) | (df_clean['idade'] > 100), 'idade'] = np.nan
df_clean.loc[df_clean['renda_mensal'] > 200000.0, 'renda_mensal'] = np.nan

# Step 3: Feature Engineering
df_clean['valor_parcela'] = df_clean['valor_solicitado'] / df_clean['numero_parcelas']
df_clean['comprometimento_renda'] = df_clean['valor_parcela'] / (df_clean['renda_mensal'] + 1e-5)

# Step 4: Imputação de nulos pela Mediana
cols_num = ['idade', 'renda_mensal', 'score_serasa', 'comprometimento_renda']
for col in cols_num:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Step 5: Remoção de colunas de ruído e IDs
df_clean = df_clean.drop(columns=['proponente_id', 'ip_origem_hash', 'ruido_estocastico', 'numero_da_sorte_app'])

# Step 6: One-Hot Encoding
df_encoded = pd.get_dummies(df_clean, columns=['tipo_vinculo', 'estado_civil', 'escolaridade'], drop_first=True)

# Step 7: Divisão Treino e Teste
X = df_encoded.drop(columns=['inadimplente'])
y = df_encoded['inadimplente']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Data Preparation Concluído com Sucesso!")
print(f"Dimensão de X_train: {X_train.shape}")
print(f"Dimensão de X_test:  {X_test.shape}")


In [ ]:
# Visualização gráfica final após Data Preparation
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.boxplot(y=df_clean['idade'], color='lightgreen')
plt.title('Idades Sanitizadas (Sem Outliers Aberrantes)')

plt.subplot(1, 2, 2)
sns.histplot(df_clean['comprometimento_renda'], kde=True, color='purple', bins=30)
plt.title('Distribuição do Comprometimento de Renda')

plt.tight_layout()
plt.show()


---
## 4. Exercícios de Fixação

1. Explique por que a imputação de valores nulos utilizando a **Mediana** é preferível à **Média** em variáveis financeiras como renda mensal.
2. Qual a utilidade da função `pd.get_dummies()` com a opção `drop_first=True`?
3. O que acontece com os dados quando definimos `stratify=y` na função `train_test_split`?
